# 98. Macro & Market Regime 데이터 구축

## 📋 개요
종목별 데이터 수집에 앞서, 한국 시장의 전역적(Global) 환경 변수인 거시 경제 지표와 Market Regime(강세/약세/보합장) 시그널을 계산하여 별도로 저장합니다.

## ✅ 주요 기능
1. **장기 시계열 확보**: 2000년부터 현재까지의 거시 데이터 수집 (Look-back 보장)
2. **PyKRX 대체**: `FinanceDataReader`를 활용한 환율, 글로벌 지수, KOSPI 수집
3. **Regime 판단**: KOSPI 200일선 및 변동성 기반의 Bull(1), Bear(-1), Neutral(0) 상태 계산

In [ ]:
import pandas as pd
import numpy as np
import FinanceDataReader as fdr
from datetime import datetime, timedelta
from pathlib import Path

# 설정 로드 및 경로 준비 (기존 config 활용)
from src.utils.config import load_config, ProjectPaths
cfg = load_config()
paths = ProjectPaths.from_config(cfg)

# 매크로 데이터를 저장할 폴더
meta_dir = Path(cfg['paths'].get('meta_dir', 'data/99_meta'))
macro_filepath = meta_dir / "macro_regime.parquet"

print(f"📁 매크로 데이터 저장 경로: {macro_filepath}")

In [ ]:
# ==========================================
# 1. 동적 날짜 설정 및 데이터 수집
# ==========================================
# 설정 파일의 시작일로부터 365일 전을 수집 시작일로 설정 (버퍼 확보)
base_start = pd.to_datetime(cfg['data_collection']['start_date'])
base_end = pd.to_datetime(cfg['data_collection']['end_date'])
fetch_start = (base_start - timedelta(days=365)).strftime('%Y-%m-%d')
fetch_end = base_end.strftime('%Y-%m-%d')

print(f"📥 데이터 수집 중... ({fetch_start} ~ {fetch_end})")

macro_raw = {}
macro_raw['kospi'] = fdr.DataReader('KS11', fetch_start, fetch_end)['Close']
macro_raw['sp500'] = fdr.DataReader('US500', fetch_start, fetch_end)['Close']
macro_raw['usd_krw'] = fdr.DataReader('USD/KRW', fetch_start, fetch_end)['Close']

try:
    macro_raw['vix'] = fdr.DataReader('FRED:VIXCLS', fetch_start, fetch_end)['VIXCLS']
except:
    macro_raw['vix'] = pd.Series(dtype=float)

df_macro = pd.DataFrame(macro_raw).ffill()
print(f"✅ 수집 완료: {len(df_macro):,} 거래일")

In [ ]:
# ==========================================
# 2. Market Regime 및 파생 피처 계산
# ==========================================
print("⚙️ Market Regime 및 캘린더 피처 계산 중...")

# 1. KOSPI 기술적 지표 계산
df_macro['kospi_ma200'] = df_macro['kospi'].rolling(window=200).mean()
df_macro['kospi_vol20'] = df_macro['kospi'].pct_change().rolling(window=20).std()

# 동적 임계값: 최근 1년(250거래일) 변동성 중위수
vol_median = df_macro['kospi_vol20'].rolling(window=250).median()

# 2. Regime 판단 로직
#   - Bull (1) : 주가가 200일선 위
#   - Bear (-1): 주가가 200일선 아래 & 단기 변동성이 장기 중위수보다 큼 (투매 장세)
#   - Neutral (0): 그 외 횡보장
conditions = [
    (df_macro['kospi'] > df_macro['kospi_ma200']),
    (df_macro['kospi'] < df_macro['kospi_ma200']) & (df_macro['kospi_vol20'] > vol_median)
]
choices = [1, -1]
df_macro['market_regime'] = np.select(conditions, choices, default=0)

# 3. 미국 시장 수익률
# 한국 시초가에 영향을 주는 전일(혹은 직전 거래일) 미국 시장 수익률
df_macro['us_return_1d'] = df_macro['sp500'].pct_change().shift(1)

# 4. 결측치 제거 (초기 250일분 이동평균 계산 구간 제거)
df_macro = df_macro.dropna()

print("✅ 계산 완료. Regime 분포:")
print(df_macro['market_regime'].value_counts(normalize=True).map('{:.1%}'.format))

In [ ]:
# ==========================================
# 3. 데이터 저장
# ==========================================
# 인덱스(Date)를 컬럼으로 리셋하여 병합하기 편하게 만듭니다.
df_macro_final = df_macro.reset_index()
if 'Date' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'Date': 'date'})
elif 'index' in df_macro_final.columns:
    df_macro_final = df_macro_final.rename(columns={'index': 'date'})

# 필요한 핵심 피처만 선택 (KOSPI MA 등은 레짐 계산용이므로 제외 가능하나 분석용으로 남겨둠)
final_cols = ['date', 'kospi', 'usd_krw', 'vix', 'us_return_1d', 'market_regime']
df_save = df_macro_final[final_cols].copy()

# Parquet 포맷으로 저장
df_save.to_parquet(macro_filepath, index=False)

print(f"💾 매크로 데이터 저장 완료! -> {macro_filepath}")
print(f"   - 총 데이터 수: {len(df_save):,}일")
display(df_save.head())